In [1]:
pip install sodapy


[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# %% [markdown]
# # 1. Ingesta (Capa Bronce)
# Convertimos CSV crudo a formato Delta Lake.

# %%
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
# Para control de la ingesta
from pyspark.sql.functions import current_timestamp, input_file_name
from delta import *
import os
from sodapy import Socrata
import pandas as pd

# Configuración Spark
# URL del Master (definida en docker-compose)
master_url = "spark://spark-master:7077"

# Configuración y Añadimos Delta Lake
# Despues de master_url
builder = SparkSession.builder \
    .appName("Lab_SECOP_Bronze") \
    .master(master_url) \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.memory", "2g") 

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# %%
# Extracción desde Socrata ultimos 100 mil registros
print("Extrayendo datos desde Socrata (SECOP II)")
client = Socrata("www.datos.gov.co", None) # Crea cliente Socrata
limit = 10000
offset = 0
total_rows = 100000
chunks = []

while offset < total_rows:
    results = client.get(
        "jbjy-vk9h",
        limit=limit,
        offset=offset,
        order="fecha_de_firma DESC" #del más reciente al más antiguo
    )
    if not results:
        break

    chunks.append(pd.DataFrame.from_records(results))
    offset += limit
    print(f"Registros extraídos: {offset}")

# Concatenar todo
pdf_raw = pd.concat(chunks, ignore_index=True)

# Convertir a Spark DataFrame
df_raw = spark.createDataFrame(pdf_raw)

print(f"Total registros cargados: {df_raw.count()}")



:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4476d5cf-4ac4-46b6-bf0c-1b2e3d336fe3;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.0.0/delta-spark_2.12-3.0.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.0.0!delta-spark_2.12.jar (771ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.0.0/delta-storage-3.0.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.0.0!delta-storage.jar (191ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (203ms)
:: resolution report :: resolve 2881ms :: artifacts dl 

Extrayendo datos desde Socrata (SECOP II)
Registros extraídos: 10000


26/02/01 16:44:28 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Registros extraídos: 20000
Registros extraídos: 30000
Registros extraídos: 40000
Registros extraídos: 50000
Registros extraídos: 60000
Registros extraídos: 70000
Registros extraídos: 80000
Registros extraídos: 90000
Registros extraídos: 100000


26/02/01 16:46:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/02/01 16:46:30 WARN TaskSetManager: Stage 0 contains a task of very large size (72371 KiB). The maximum recommended task size is 1000 KiB.
[Stage 0:>                                                          (0 + 2) / 2]

Total registros cargados: 100000


In [4]:
# Normalización técnica de nombres de columnas
# (REQUERIDO por Delta Lake)
nuevas_columnas = [
    c.strip()
     .replace(" ", "_")
     .replace("(", "")
     .replace(")", "")
     .replace(".", "")
     .replace(",", "")
     .lower()
    for c in df_raw.columns
]
# Aplicamos los nuevos nombres al DataFrame
df_raw = df_raw.toDF(*nuevas_columnas)

# ESCRITURA BRONCE (Delta)
# Agregar variables de control de la ingesta

df_bronze = (
    df_raw
    .withColumn("fecha_ingesta", current_timestamp())
    .withColumn("archivo_origen", input_file_name())
)

# --------------------------------------------
# Escritura en capa Bronze (Delta Lake)
# --------------------------------------------

print("Escribiendo datos en capa Bronze...")
output_path = "/app/data/lakehouse/bronze/secop"

(
    df_bronze.write
    .format("delta")
    .mode("append")   # No borra nada y mantiene el historial delta
    .save(output_path)
)

print(f"Ingesta Bronze completada. Registros procesados: {df_bronze.count()}")


Escribiendo datos en capa Bronze...


26/02/01 16:50:15 WARN TaskSetManager: Stage 4 contains a task of very large size (72371 KiB). The maximum recommended task size is 1000 KiB.
26/02/01 16:50:27 WARN TaskSetManager: Stage 12 contains a task of very large size (72371 KiB). The maximum recommended task size is 1000 KiB.
[Stage 12:>                                                         (0 + 2) / 2]

Ingesta Bronze completada. Registros procesados: 100000


In [5]:
# Validación Delta Lake (historial)
print("Historial Delta - Bronze:")
spark.sql(
    f"DESCRIBE HISTORY delta.`{output_path}`"
).show(truncate=False)

Historial Delta - Bronze:


+-------+-----------------------+------+--------+---------+-----------------------------------+----+--------+---------+-----------+--------------+-------------+--------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation|operationParameters                |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                    |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------+-----------------------------------+----+--------+---------+-----------+--------------+-------------+--------------------------------------------------------------------+------------+-----------------------------------+
|0      |2026-02-01 16:50:22.711|NULL  |NULL    |WRITE    |{mode -> Append, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable  |true         |{nu

In [7]:
spark.stop()